# Metodologia Design Science Research (DSR)

**Etapa de Pesquisa (Peffers et al., 2007):**
### 3. Design e Desenvolvimento (Design and Development)

**Objetivo Acadêmico:** Este notebook transforma a Base Mestra em um conjunto de preditores de alta dimensão (*Features*). A engenharia de variáveis como *Lags* (atrasos temporais) e Médias Móveis é essencial para capturar a dependência serial da demanda. Além disso, a codificação trigonométrica (Seno/Cosseno) para variáveis cíclicas como o dia da semana garante que o modelo compreenda o comportamento periódico do campus, alinhando o artefato às necessidades técnicas discutidas por Rodrigues et al. (2024).


# 04 - Engenharia de Features (Lags e Sazonalidade)
Este notebook cria as variáveis preditivas sobre a Base Mestra filtrada, garantindo ausência de vazamento de dados.

In [1]:
import pandas as pd
import numpy as np
import os

# 1. Carregamento
BASE_MESTRA = '../data/base_mestra_consolidada.csv'
SAIDA_FEATURES = '../data/base_features_final.csv'

df = pd.read_csv(BASE_MESTRA)
df['data'] = pd.to_datetime(df['data'])
df = df.sort_values('data').reset_index(drop=True)

# 2. Features de Calendário
df['dia_semana'] = df['data'].dt.dayofweek
df['dia_semana_sin'] = np.sin(2 * np.pi * df['dia_semana'] / 7)
df['dia_semana_cos'] = np.cos(2 * np.pi * df['dia_semana'] / 7)
df['eh_segunda'] = (df['dia_semana'] == 0).astype(int)
df['eh_sexta'] = (df['dia_semana'] == 4).astype(int)

# 3. Lags de Reservas (Anti-Leakage)
lags_reserva = [1, 2, 3, 7]
for lag in lags_reserva:
    df[f'total_reservas_lag_{lag}'] = df['total_reservas'].shift(lag)

# Médias Móveis (Shifted to avoid leakage)
df['reservas_media_movel_3d'] = df['total_reservas'].shift(1).rolling(window=3).mean()
df['reservas_media_movel_5d'] = df['total_reservas'].shift(1).rolling(window=5).mean()

# 4. Clima Lagged
df['chuva_ontem'] = df['chuva_soma'].shift(1)
df['temp_media_ontem'] = df['temp_media'].shift(1)

# Interação
df['interacao_reserva_evento'] = df['total_reservas'] * df['eh_evento_especial']

# 5. Encoding de Etapas
if 'etapa' in df.columns:
    df = pd.get_dummies(df, columns=['etapa'], prefix='etapa_acad', drop_first=False)
    # Convert to int
    for c in df.columns:
        if 'etapa_acad' in c: df[c] = df[c].astype(int)

# 6. Limpeza de Lags Nulos e Exportação
df_features = df.dropna().copy()
df_features.to_csv(SAIDA_FEATURES, index=False)

print(f"✨ Engenharia concluída sobre base filtrada.")
print(f"📊 Shape: {df_features.shape}")
print(f"📈 Proteínas na base: {df_features['proteina_principal'].unique()}")

✨ Engenharia concluída sobre base filtrada.
📊 Shape: (92, 37)
📈 Proteínas na base: ['SUINA' 'AVE' 'BOVINA' 'FEIJOADA' 'PEIXE']
